# 🏥 HEALTHCARE INSIGHT360
## AI-Powered Healthcare Analytics & Patient Intelligence Dashboard
**Subtitle:** Transforming Healthcare Data into Actionable Patient, Operational & Clinical Insights

---
**Author:** Mansi Kushwaha  
**Project:** Healthcare Insight360  
**Dataset:** healthcare_dataset.csv (Synthetic / Anonymized)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Scikit-learn, Streamlit  

---
> ⚠️ **Disclaimer:** All analyses, predictions, and models in this notebook are strictly for **educational and analytical purposes only**. They do NOT constitute medical advice, clinical decision support, or medical diagnosis.

## 📦 PART 0 — Library Imports & Configuration

In [ ]:
# ============================================================
# PART 0 — IMPORTS & GLOBAL CONFIGURATION
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

SEED = 42
np.random.seed(SEED)

# Output directories
for d in ['../outputs/charts', '../outputs/cleaned_data', '../outputs/reports', '../models']:
    os.makedirs(d, exist_ok=True)

print('✅ All libraries imported successfully.')
print(f'📁 Output directories ready.')

## 📂 PART 1 — Data Loading & Initial Exploration

In [ ]:
# ============================================================
# PART 1 — DATA LOADING
# ============================================================
DATA_PATH = '../data/healthcare_dataset.csv'

def load_dataset(path):
    """Load CSV or Excel dataset based on file extension."""
    ext = os.path.splitext(path)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(path)
    elif ext in ['.xlsx', '.xls']:
        df = pd.read_excel(path, engine='openpyxl')
    else:
        raise ValueError(f'Unsupported file format: {ext}')
    print(f'✅ Dataset loaded: {path}')
    print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
    return df

df_raw = load_dataset(DATA_PATH)

print('\n--- First 5 Rows ---')
df_raw.head()

In [ ]:
print('--- Column Information ---')
df_raw.info()
print('\n--- Descriptive Statistics ---')
df_raw.describe(include='all')

In [ ]:
# ---- Data Quality Report ----
def data_quality_report(df):
    """Generate a comprehensive data quality report."""
    report = pd.DataFrame({
        'Column': df.columns,
        'Dtype': df.dtypes.values,
        'Non-Null Count': df.notnull().sum().values,
        'Null Count': df.isnull().sum().values,
        'Null %': (df.isnull().mean() * 100).round(2).values,
        'Unique Values': df.nunique().values,
        'Sample Values': [df[c].dropna().unique()[:3].tolist() for c in df.columns]
    })
    print('\n=== DATA QUALITY REPORT ===')
    print(f'Total rows        : {df.shape[0]:,}')
    print(f'Total columns     : {df.shape[1]}')
    print(f'Duplicate rows    : {df.duplicated().sum():,}')
    print(f'Total missing vals: {df.isnull().sum().sum():,}')
    print()
    return report

dq = data_quality_report(df_raw)
dq

In [ ]:
# ---- Outlier Detection for Billing Amount ----
Q1 = df_raw['Billing Amount'].quantile(0.25)
Q3 = df_raw['Billing Amount'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_raw[(df_raw['Billing Amount'] < Q1 - 1.5 * IQR) |
                   (df_raw['Billing Amount'] > Q3 + 1.5 * IQR)]
print(f'Billing Amount outliers (IQR method): {len(outliers):,} rows')
print(f'Billing Amount range: ${df_raw["Billing Amount"].min():,.2f} — ${df_raw["Billing Amount"].max():,.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_raw['Billing Amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Billing Amount Distribution (Raw)', fontsize=13)
axes[0].set_xlabel('Billing Amount ($)')
axes[1].boxplot(df_raw['Billing Amount'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightcoral'))
axes[1].set_title('Billing Amount Box Plot', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/charts/01_billing_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧹 PART 2 — Data Cleaning & Feature Engineering

In [ ]:
# ============================================================
# PART 2 — DATA CLEANING
# ============================================================
df = df_raw.copy()

# 1. Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed : {before - len(df):,}')

# 2. Standardize column names
df.columns = [c.strip().replace(' ', '_') for c in df.columns]
print('Columns standardized:', df.columns.tolist())

# 3. Standardize string columns to Title Case
str_cols = ['Name', 'Gender', 'Blood_Type', 'Medical_Condition',
            'Doctor', 'Hospital', 'Insurance_Provider',
            'Admission_Type', 'Medication', 'Test_Results']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title()

# 4. Clean hospital names (remove trailing commas/spaces)
df['Hospital'] = df['Hospital'].str.rstrip(',').str.strip()

# 5. Convert date columns
df['Date_of_Admission'] = pd.to_datetime(df['Date_of_Admission'], errors='coerce')
df['Discharge_Date']    = pd.to_datetime(df['Discharge_Date'], errors='coerce')
invalid_dates = df['Date_of_Admission'].isna().sum() + df['Discharge_Date'].isna().sum()
print(f'Invalid date rows   : {invalid_dates}')

# Drop rows with unparseable dates
df.dropna(subset=['Date_of_Admission', 'Discharge_Date'], inplace=True)

# 6. Validate age (0–120)
before = len(df)
df = df[(df['Age'] >= 0) & (df['Age'] <= 120)]
print(f'Impossible ages removed: {before - len(df):,}')

# 7. Handle Billing Amount outliers — cap at 99th percentile
cap99 = df['Billing_Amount'].quantile(0.99)
low01 = df['Billing_Amount'].quantile(0.01)
df['Billing_Amount'] = df['Billing_Amount'].clip(lower=low01, upper=cap99)
print(f'Billing Amount capped at 1st–99th percentile: ${low01:,.2f} — ${cap99:,.2f}')

print(f'\n✅ Cleaned dataset shape: {df.shape}')

In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

# Length of Stay (days)
df['Length_of_Stay'] = (df['Discharge_Date'] - df['Date_of_Admission']).dt.days
df['Length_of_Stay'] = df['Length_of_Stay'].clip(lower=0)  # no negative stays

# Admission temporal features
df['Admission_Year']  = df['Date_of_Admission'].dt.year
df['Admission_Month'] = df['Date_of_Admission'].dt.month
df['Admission_Day']   = df['Date_of_Admission'].dt.day
df['Admission_Month_Name'] = df['Date_of_Admission'].dt.strftime('%b')
df['Admission_Quarter'] = df['Date_of_Admission'].dt.quarter

# Discharge temporal features
df['Discharge_Month'] = df['Discharge_Date'].dt.month
df['Discharge_Year']  = df['Discharge_Date'].dt.year

# Age Group
bins   = [0, 17, 35, 50, 65, 80, 120]
labels = ['<18', '18-35', '36-50', '51-65', '66-80', '80+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)

# Revenue Category based on billing quartiles
q_labels = ['Low', 'Medium', 'High', 'Very High']
df['Revenue_Category'] = pd.qcut(df['Billing_Amount'], q=4, labels=q_labels)

# High-Risk Flag: Emergency admission OR Abnormal test result OR Long stay (>14 days)
df['High_Risk_Flag'] = (
    (df['Admission_Type'] == 'Emergency') |
    (df['Test_Results'] == 'Abnormal') |
    (df['Length_of_Stay'] > 14)
).astype(int)

# Readmission proxy: Test_Results == 'Inconclusive' treated as potential readmission indicator
# Since the dataset has no explicit Readmission column, we derive it from Test_Results
df['Readmission'] = (df['Test_Results'] == 'Inconclusive').astype(int)

# Year-Month period for time-series
df['YearMonth'] = df['Date_of_Admission'].dt.to_period('M').astype(str)

print('✅ Feature engineering complete.')
print(df[['Age_Group', 'Length_of_Stay', 'Revenue_Category', 'High_Risk_Flag', 'Readmission']].head(10))

In [ ]:
# Save cleaned dataset
df.to_csv('../outputs/cleaned_data/healthcare_cleaned.csv', index=False)
print('✅ Cleaned data saved to ../outputs/cleaned_data/healthcare_cleaned.csv')
df.head()

## 📊 PART 3 — Exploratory Data Analysis (EDA)

### 3.1 Patient Demographics

In [ ]:
# ---- Age Distribution ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age histogram
axes[0].hist(df['Age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Age Group distribution
age_grp = df['Age_Group'].value_counts().sort_index()
axes[1].bar(age_grp.index.astype(str), age_grp.values, color='teal', edgecolor='white')
axes[1].set_title('Patients by Age Group', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')

# Gender distribution
gender_counts = df['Gender'].value_counts()
axes[2].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#4e8cff', '#ff6b6b'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('Gender Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/charts/02_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Blood Type & Insurance ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bt = df['Blood_Type'].value_counts()
axes[0].bar(bt.index, bt.values, color=sns.color_palette('Set2', len(bt)))
axes[0].set_title('Blood Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

ins = df['Insurance_Provider'].value_counts()
axes[1].barh(ins.index, ins.values, color=sns.color_palette('Set3', len(ins)))
axes[1].set_title('Patients by Insurance Provider', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../outputs/charts/03_blood_insurance.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Medical Analysis

In [ ]:
# ---- Medical Conditions ----
cond_counts = df['Medical_Condition'].value_counts()

fig = px.bar(x=cond_counts.values, y=cond_counts.index, orientation='h',
             color=cond_counts.values, color_continuous_scale='Blues',
             title='Most Common Medical Conditions',
             labels={'x': 'Patient Count', 'y': 'Medical Condition'})
fig.update_layout(height=400, yaxis={'categoryorder': 'total ascending'})
fig.show()

# ---- Medical Condition by Gender ----
cond_gender = df.groupby(['Medical_Condition', 'Gender']).size().reset_index(name='Count')
fig2 = px.bar(cond_gender, x='Medical_Condition', y='Count', color='Gender',
              barmode='group', title='Medical Condition by Gender',
              color_discrete_sequence=['#4e8cff', '#ff6b6b'])
fig2.update_layout(height=400)
fig2.show()

In [ ]:
# ---- Medical Condition by Age Group ----
cond_age = df.groupby(['Medical_Condition', 'Age_Group']).size().reset_index(name='Count')
cond_age['Age_Group'] = cond_age['Age_Group'].astype(str)

fig = px.bar(cond_age, x='Medical_Condition', y='Count', color='Age_Group',
             barmode='stack', title='Medical Condition by Age Group',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=450)
fig.show()

# ---- Medication Usage ----
med = df['Medication'].value_counts()
fig2 = px.pie(values=med.values, names=med.index, title='Medication Distribution',
              hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig2.show()

### 3.3 Hospital Operations

In [ ]:
# ---- Monthly Admissions Trend ----
monthly = df.groupby('YearMonth').size().reset_index(name='Admissions')
monthly = monthly.sort_values('YearMonth')

fig = px.line(monthly, x='YearMonth', y='Admissions',
              title='Monthly Patient Admissions Trend',
              markers=True, line_shape='spline',
              color_discrete_sequence=['#3b82d4'])
fig.update_xaxes(tickangle=45, nticks=20)
fig.update_layout(height=400)
fig.show()

In [ ]:
# ---- Admission Type Distribution ----
adm_type = df['Admission_Type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']
axes[0].bar(adm_type.index, adm_type.values, color=colors, edgecolor='white')
axes[0].set_title('Admission Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Average Length of Stay by Admission Type
los_type = df.groupby('Admission_Type')['Length_of_Stay'].mean().sort_values(ascending=False)
axes[1].bar(los_type.index, los_type.values, color=['#9b59b6', '#e67e22', '#1abc9c'], edgecolor='white')
axes[1].set_title('Avg Length of Stay by Admission Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Days')

plt.tight_layout()
plt.savefig('../outputs/charts/04_admissions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Top 15 Hospitals by Patient Volume ----
top_hosp = df['Hospital'].value_counts().head(15).reset_index()
top_hosp.columns = ['Hospital', 'Patients']

fig = px.bar(top_hosp, x='Patients', y='Hospital', orientation='h',
             color='Patients', color_continuous_scale='Teal',
             title='Top 15 Hospitals by Patient Volume')
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

### 3.4 Financial Analysis

In [ ]:
# ---- Billing by Medical Condition ----
bill_cond = df.groupby('Medical_Condition')['Billing_Amount'].agg(['mean', 'sum']).reset_index()
bill_cond.columns = ['Medical_Condition', 'Avg_Billing', 'Total_Billing']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Avg Billing by Condition', 'Total Revenue by Condition'])
fig.add_trace(go.Bar(x=bill_cond['Medical_Condition'], y=bill_cond['Avg_Billing'],
                     marker_color='#3b82d4', name='Avg Billing'), row=1, col=1)
fig.add_trace(go.Bar(x=bill_cond['Medical_Condition'], y=bill_cond['Total_Billing'],
                     marker_color='#7c5cd8', name='Total Revenue'), row=1, col=2)
fig.update_layout(height=420, title_text='Financial Analysis by Medical Condition', showlegend=False)
fig.show()

In [ ]:
# ---- Insurance-wise Billing ----
ins_bill = df.groupby('Insurance_Provider')['Billing_Amount'].mean().sort_values(ascending=False)

fig = px.bar(ins_bill.reset_index(), x='Insurance_Provider', y='Billing_Amount',
             color='Billing_Amount', color_continuous_scale='Oranges',
             title='Average Billing Amount by Insurance Provider',
             labels={'Billing_Amount': 'Avg Billing ($)', 'Insurance_Provider': 'Insurance'})
fig.update_layout(height=400)
fig.show()

In [ ]:
# ---- Monthly Revenue Trend ----
monthly_rev = df.groupby('YearMonth')['Billing_Amount'].sum().reset_index()
monthly_rev.columns = ['YearMonth', 'Revenue']
monthly_rev = monthly_rev.sort_values('YearMonth')

fig = px.area(monthly_rev, x='YearMonth', y='Revenue',
              title='Monthly Revenue Trend ($)',
              color_discrete_sequence=['#7c5cd8'],
              labels={'Revenue': 'Total Revenue ($)'})
fig.update_xaxes(tickangle=45, nticks=20)
fig.update_layout(height=400)
fig.show()

### 3.5 Outcome & Readmission Analysis

In [ ]:
# ---- Test Results Distribution ----
test_res = df['Test_Results'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#e74c3c', '#f39c12']
axes[0].bar(test_res.index, test_res.values, color=colors, edgecolor='white')
axes[0].set_title('Test Results Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Length of Stay distribution
axes[1].hist(df['Length_of_Stay'], bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(df['Length_of_Stay'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["Length_of_Stay"].mean():.1f} days')
axes[1].set_title('Length of Stay Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Days')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/charts/05_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Heatmap: Admissions by Month & Year ----
heat_df = df.groupby(['Admission_Year', 'Admission_Month']).size().reset_index(name='Admissions')
heat_pivot = heat_df.pivot(index='Admission_Year', columns='Admission_Month', values='Admissions').fillna(0)

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
heat_pivot.columns = [month_names[i-1] for i in heat_pivot.columns]

plt.figure(figsize=(14, 5))
sns.heatmap(heat_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Admissions'})
plt.title('Patient Admissions Heatmap (Year × Month)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/06_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 📈 PART 4 — KPI Engine

In [ ]:
# ============================================================
# PART 4 — KPI ENGINE
# ============================================================
def compute_kpis(df):
    """Compute all healthcare KPIs from the cleaned dataframe."""
    kpis = {}

    kpis['Total Patients']            = len(df)
    kpis['Total Admissions']          = len(df)
    kpis['Total Discharges']          = int(df['Discharge_Date'].notna().sum())
    kpis['Active Patients']           = int((df['Discharge_Date'] >= pd.Timestamp.today()).sum())
    kpis['Average Patient Age']       = round(df['Age'].mean(), 1)
    kpis['Avg Length of Stay (days)'] = round(df['Length_of_Stay'].mean(), 1)
    kpis['Total Healthcare Revenue']  = round(df['Billing_Amount'].sum(), 2)
    kpis['Average Billing Amount']    = round(df['Billing_Amount'].mean(), 2)
    kpis['Readmission Rate (%)']      = round(df['Readmission'].mean() * 100, 2)
    kpis['Abnormal Outcome Rate (%)'] = round((df['Test_Results'] == 'Abnormal').mean() * 100, 2)
    kpis['Most Common Condition']     = df['Medical_Condition'].mode()[0]
    kpis['Most Common Medication']    = df['Medication'].mode()[0]
    kpis['Emergency Admission (%)']   = round((df['Admission_Type'] == 'Emergency').mean() * 100, 2)
    kpis['High-Risk Patients (%)']    = round(df['High_Risk_Flag'].mean() * 100, 2)
    kpis['Top Insurance Provider']    = df['Insurance_Provider'].mode()[0]

    # Month with highest admissions
    top_month = df['Admission_Month_Name'].value_counts().idxmax()
    kpis['Peak Admission Month']      = top_month

    # Hospital with highest patient volume
    kpis['Busiest Hospital']          = df['Hospital'].value_counts().idxmax()

    # Average treatment cost (=avg billing)
    kpis['Avg Treatment Cost']        = kpis['Average Billing Amount']

    return kpis

kpis = compute_kpis(df)

print('=' * 55)
print('        HEALTHCARE INSIGHT360 — KPI DASHBOARD')
print('=' * 55)
for k, v in kpis.items():
    if isinstance(v, float) and v > 1000:
        print(f'{k:<35}: ${v:>15,.2f}')
    elif isinstance(v, float):
        print(f'{k:<35}: {v:>15.2f}')
    else:
        print(f'{k:<35}: {str(v):>15}')
print('=' * 55)

## 🔬 PART 5 — Advanced Analytics

### 5.1 Correlation Analysis

In [ ]:
# ---- Correlation Heatmap ----
num_cols = ['Age', 'Billing_Amount', 'Length_of_Stay', 'Readmission', 'High_Risk_Flag',
            'Admission_Month', 'Admission_Year']
corr = df[num_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Key Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/07_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Scatter: Age vs Billing ----
fig = px.scatter(df, x='Age', y='Billing_Amount', color='Medical_Condition',
                 opacity=0.6, title='Age vs Billing Amount (by Medical Condition)',
                 labels={'Billing_Amount': 'Billing Amount ($)'},
                 color_discrete_sequence=px.colors.qualitative.Set1)
fig.update_layout(height=450)
fig.show()

### 5.2 Patient Segmentation (K-Means Clustering)

In [ ]:
# ---- K-Means Patient Segmentation ----
seg_features = ['Age', 'Billing_Amount', 'Length_of_Stay']
seg_df = df[seg_features].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(seg_df)

# Elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.savefig('../outputs/charts/08_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

# Fit with K=4
K_OPT = 4
km_final = KMeans(n_clusters=K_OPT, random_state=SEED, n_init=10)
seg_df['Cluster'] = km_final.fit_predict(X_scaled)
df['Patient_Segment'] = km_final.predict(scaler.transform(df[seg_features].fillna(df[seg_features].mean())))

seg_summary = seg_df.groupby('Cluster')[seg_features].mean().round(1)
seg_summary['Patient_Count'] = seg_df.groupby('Cluster').size()
print('Patient Segmentation Summary (K=4):')
print(seg_summary)

# Segment labels
seg_labels = {
    seg_summary['Age'].idxmin(): 'Young-Low Cost',
}
sorted_by_billing = seg_summary.sort_values('Billing_Amount')
segment_names = ['Low Utilization', 'Moderate Care', 'High Utilization', 'Complex/Critical']
seg_name_map = {idx: name for idx, name in zip(sorted_by_billing.index, segment_names)}
df['Segment_Label'] = df['Patient_Segment'].map(seg_name_map)
print('\nSegment Labels:', seg_name_map)

In [ ]:
# ---- Visualize Segments ----
fig = px.scatter(seg_df, x='Age', y='Billing_Amount', color='Cluster',
                 size='Length_of_Stay', opacity=0.7,
                 title='Patient Segments (K-Means Clustering)',
                 labels={'Billing_Amount': 'Billing Amount ($)', 'Cluster': 'Segment'},
                 color_continuous_scale='Viridis')
fig.update_layout(height=450)
fig.show()

### 5.3 Time-Series Analysis

In [ ]:
# ---- Monthly Admissions + Revenue + Readmissions ----
ts = df.groupby('YearMonth').agg(
    Admissions=('Age', 'count'),
    Revenue=('Billing_Amount', 'sum'),
    Readmissions=('Readmission', 'sum'),
    Avg_LOS=('Length_of_Stay', 'mean')
).reset_index().sort_values('YearMonth')

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Monthly Admissions', 'Monthly Revenue ($)',
                                    'Monthly Readmissions', 'Avg Length of Stay'])
fig.add_trace(go.Scatter(x=ts['YearMonth'], y=ts['Admissions'], mode='lines+markers',
                          line=dict(color='#3b82d4', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=ts['YearMonth'], y=ts['Revenue'], mode='lines',
                          fill='tozeroy', line=dict(color='#7c5cd8', width=2)), row=1, col=2)
fig.add_trace(go.Bar(x=ts['YearMonth'], y=ts['Readmissions'],
                     marker_color='#e74c3c'), row=2, col=1)
fig.add_trace(go.Scatter(x=ts['YearMonth'], y=ts['Avg_LOS'], mode='lines+markers',
                          line=dict(color='#2ecc71', width=2)), row=2, col=2)
fig.update_layout(height=600, title_text='Time-Series Analysis — Key Healthcare Metrics',
                  showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()

## 🤖 PART 6 — Machine Learning: Readmission Risk Prediction

In [ ]:
# ============================================================
# PART 6 — MACHINE LEARNING
# ============================================================
# ⚠️  DISCLAIMER: This model is for educational/analytical purposes ONLY.
#     It is NOT a medical diagnosis system and must NOT be used
#     for clinical decision-making.
# ============================================================

print('⚠️  DISCLAIMER: This model is for educational/analytical purposes ONLY.')
print('    It is NOT a medical diagnosis or clinical decision-making system.\n')

# Features & Target
ML_FEATURES = ['Age', 'Billing_Amount', 'Length_of_Stay',
                'Gender', 'Medical_Condition', 'Admission_Type',
                'Insurance_Provider', 'Medication', 'Test_Results',
                'Blood_Type', 'Age_Group']
TARGET = 'Readmission'

ml_df = df[ML_FEATURES + [TARGET]].dropna().copy()
ml_df['Age_Group'] = ml_df['Age_Group'].astype(str)

# Numeric & categorical feature split
num_features = ['Age', 'Billing_Amount', 'Length_of_Stay']
cat_features = [f for f in ML_FEATURES if f not in num_features]

X = ml_df[ML_FEATURES]
y = ml_df[TARGET]

print(f'Dataset: {len(X):,} samples | Target distribution:')
print(y.value_counts(normalize=True).rename({0: 'Not Readmitted', 1: 'Readmitted'}).mul(100).round(2).to_string())

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'\nTrain: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# ---- Build Preprocessing Pipeline ----
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

# ---- Train Multiple Models ----
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=SEED),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=SEED),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = {
        'pipeline': pipe,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_proba),
        'y_pred':    y_pred,
        'y_proba':   y_proba,
    }
    print(f'[{name}] Acc={results[name]["Accuracy"]:.3f} | '
          f'P={results[name]["Precision"]:.3f} | '
          f'R={results[name]["Recall"]:.3f} | '
          f'F1={results[name]["F1"]:.3f} | '
          f'AUC={results[name]["ROC-AUC"]:.3f}')

In [ ]:
# ---- Model Comparison Chart ----
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy':  [results[m]['Accuracy']  for m in results],
    'Precision': [results[m]['Precision'] for m in results],
    'Recall':    [results[m]['Recall']    for m in results],
    'F1':        [results[m]['F1']        for m in results],
    'ROC-AUC':   [results[m]['ROC-AUC']   for m in results],
})

fig = px.bar(metrics_df.melt(id_vars='Model', var_name='Metric', value_name='Score'),
             x='Model', y='Score', color='Metric', barmode='group',
             title='Model Performance Comparison',
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(height=450, yaxis_range=[0, 1])
fig.show()

# Select best model by ROC-AUC
best_model_name = max(results, key=lambda x: results[x]['ROC-AUC'])
best_model = results[best_model_name]['pipeline']
print(f'\n✅ Best model: {best_model_name} (AUC={results[best_model_name]["ROC-AUC"]:.3f})')

In [ ]:
# ---- Confusion Matrix (Best Model) ----
y_pred_best = results[best_model_name]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'])
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curves (all models)
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={res["ROC-AUC"]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/charts/09_model_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Feature Importance (Random Forest) ----
rf_pipe = results['Random Forest']['pipeline']
rf_clf  = rf_pipe.named_steps['clf']
ohe_names = rf_pipe.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_features)
feature_names = num_features + list(ohe_names)
importances = rf_clf.feature_importances_

fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

fig = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
             color='Importance', color_continuous_scale='Blues',
             title='Top 15 Feature Importances — Random Forest')
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

# Save best model
joblib.dump(best_model, '../models/readmission_model.pkl')
joblib.dump(rf_pipe, '../models/random_forest_model.pkl')
print(f'✅ Models saved to ../models/')

In [ ]:
# ---- Classification Report ----
print(f'=== Classification Report: {best_model_name} ===')
print(classification_report(y_test, y_pred_best,
                             target_names=['Not Readmitted', 'Readmitted']))

## 💡 PART 7 — Automated Insights Engine

In [ ]:
# ============================================================
# PART 7 — AUTOMATED INSIGHTS ENGINE
# ============================================================
def generate_insights(df, kpis):
    """Dynamically generate data-driven healthcare insights."""
    insights = []

    # 1. Most common condition
    top_cond  = df['Medical_Condition'].value_counts()
    top_c_pct = top_cond.iloc[0] / len(df) * 100
    insights.append(f"📌 '{top_cond.index[0]}' is the most common medical condition, "
                    f"representing {top_c_pct:.1f}% of all patients ({top_cond.iloc[0]:,}).")

    # 2. Emergency admission
    insights.append(f"🚨 Emergency admissions account for {kpis['Emergency Admission (%)']:.1f}% of all admissions.")

    # 3. Readmission
    insights.append(f"🔄 The overall readmission rate (inconclusive test results proxy) is "
                    f"{kpis['Readmission Rate (%)']:.1f}%.")

    # 4. High-risk patients
    insights.append(f"⚠️  {kpis['High-Risk Patients (%)']:.1f}% of patients are classified as high-risk "
                    f"(Emergency admission, Abnormal test result, or LOS > 14 days).")

    # 5. Revenue
    insights.append(f"💰 Total healthcare revenue in the dataset: "
                    f"${kpis['Total Healthcare Revenue']:,.0f} | Avg billing: ${kpis['Average Billing Amount']:,.2f}.")

    # 6. Avg length of stay
    insights.append(f"🛏️  Average length of hospital stay: {kpis['Avg Length of Stay (days)']:.1f} days.")

    # 7. Age insight
    top_age = df['Age_Group'].astype(str).value_counts().idxmax()
    top_age_pct = df['Age_Group'].astype(str).value_counts().max() / len(df) * 100
    insights.append(f"👥 Age group '{top_age}' has the highest patient volume ({top_age_pct:.1f}% of total).")

    # 8. Gender
    gend = df['Gender'].value_counts()
    insights.append(f"⚧ Gender split: {gend.index[0]} ({gend.iloc[0]/len(df)*100:.1f}%) vs "
                    f"{gend.index[1]} ({gend.iloc[1]/len(df)*100:.1f}%).")

    # 9. Top hospital
    insights.append(f"🏥 Busiest hospital: '{kpis['Busiest Hospital']}' by patient volume.")

    # 10. Trend: month
    insights.append(f"📅 Peak admission month: {kpis['Peak Admission Month']}.")

    # 11. Top insurance
    top_ins = df['Insurance_Provider'].value_counts()
    insights.append(f"🛡️  '{top_ins.index[0]}' is the most common insurance provider "
                    f"({top_ins.iloc[0]/len(df)*100:.1f}% of patients).")

    # 12. LOS by condition
    los_cond = df.groupby('Medical_Condition')['Length_of_Stay'].mean().sort_values(ascending=False)
    insights.append(f"📊 '{los_cond.index[0]}' has the longest average hospital stay "
                    f"({los_cond.iloc[0]:.1f} days).")

    return insights

insights = generate_insights(df, kpis)
print('\n=== AUTOMATED HEALTHCARE INSIGHTS ===')
for i, insight in enumerate(insights, 1):
    print(f'{i:02d}. {insight}')

## 📊 PART 8 — Summary & Final Export

In [ ]:
# ---- Export KPIs to CSV ----
kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Value'])
kpi_df.to_csv('../outputs/reports/kpi_summary.csv', index=False)
print('✅ KPI summary saved to ../outputs/reports/kpi_summary.csv')

# ---- Export Insights ----
with open('../outputs/reports/automated_insights.txt', 'w') as f:
    f.write('HEALTHCARE INSIGHT360 — AUTOMATED INSIGHTS\n')
    f.write('=' * 55 + '\n')
    for i, ins in enumerate(insights, 1):
        f.write(f'{i:02d}. {ins}\n')
print('✅ Insights saved to ../outputs/reports/automated_insights.txt')

# ---- Final Summary ----
print('\n======================================')
print(' HEALTHCARE INSIGHT360 — COMPLETE ✅')
print('======================================')
print(f' Total Patients   : {kpis["Total Patients"]:,}')
print(f' Total Revenue    : ${kpis["Total Healthcare Revenue"]:,.0f}')
print(f' Readmission Rate : {kpis["Readmission Rate (%)"]:.1f}%')
print(f' Best ML Model    : {best_model_name}')
print(f' Best AUC         : {results[best_model_name]["ROC-AUC"]:.3f}')
print('======================================')